In [1]:
# =========================
# CELL 0
# Imports + file matching + utility functions
# =========================
import os
import re
import glob
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# -------------------------
# USER CONFIG
# -------------------------
# Option A: all files in one folder, e.g. ~/Downloads
USE_ONE_FOLDER = True
BASE_DIR = os.path.expanduser("/Users/skyxu/Downloads")

# Option B: three separate folders
CUTOFF_DIR = os.path.expanduser("/Users/skyxu/Downloads/cutoff_curves")
FULL_DIR   = os.path.expanduser("/Users/skyxu/Downloads/full_curves")
PARAMS_DIR = os.path.expanduser("/Users/skyxu/Downloads/params_files")

# Filtering
YEAR = 2024        # e.g. 2018 ; use None for all years
LIMIT = None         # e.g. 10, 50 ; use None for all matched events

# Training
NUM_EPOCHS = 3000
LR = 0.01
VERBOSE_EVERY = 10

# Output
OUTPUT_DIR = os.path.expanduser("~/Downloads")
MAIN_TABLE_CSV = os.path.join(OUTPUT_DIR, "parameter_comparison_table.csv")
OVERALL_SUMMARY_CSV = os.path.join(OUTPUT_DIR, "parameter_summary_overall.csv")
YEAR_SUMMARY_CSV = os.path.join(OUTPUT_DIR, "parameter_summary_by_year.csv")


def clean_name(path):
    base = os.path.basename(path)
    base = re.sub(r"\.dat$", "", base)
    base = re.sub(r"\(\d+\)$", "", base)
    return base


def extract_event_key(path):
    name = clean_name(path)

    m = re.match(r"curve_(\d{4})_(\d+)_cutoff$", name)
    if m:
        return int(m.group(1)), m.group(2)

    m = re.match(r"curve_(\d{4})_(\d+)$", name)
    if m:
        return int(m.group(1)), m.group(2)

    m = re.match(r"params_(\d{4})_(\d+)$", name)
    if m:
        return int(m.group(1)), m.group(2)

    return None


def build_event_triplets(use_one_folder=True, base_dir=None,
                         cutoff_dir=None, full_dir=None, params_dir=None,
                         year=None, limit=None):
    if use_one_folder:
        all_files = glob.glob(os.path.join(base_dir, "*.dat"))
        cutoff_files = [f for f in all_files if "_cutoff" in clean_name(f)]
        full_files   = [f for f in all_files if clean_name(f).startswith("curve_") and "_cutoff" not in clean_name(f)]
        params_files = [f for f in all_files if clean_name(f).startswith("params_")]
    else:
        cutoff_files = glob.glob(os.path.join(cutoff_dir, "*.dat"))
        full_files   = glob.glob(os.path.join(full_dir, "*.dat"))
        params_files = glob.glob(os.path.join(params_dir, "*.dat"))

    cutoff_map = {}
    full_map = {}
    params_map = {}

    for f in cutoff_files:
        key = extract_event_key(f)
        if key is not None:
            cutoff_map[key] = f

    for f in full_files:
        key = extract_event_key(f)
        if key is not None:
            full_map[key] = f

    for f in params_files:
        key = extract_event_key(f)
        if key is not None:
            params_map[key] = f

    common_keys = sorted(set(cutoff_map) & set(full_map) & set(params_map))

    if year is not None:
        common_keys = [k for k in common_keys if k[0] == int(year)]

    if limit is not None:
        common_keys = common_keys[:limit]

    triplets = []
    for (yr, eid) in common_keys:
        triplets.append({
            "year": yr,
            "event_id": eid,
            "cutoff_file": cutoff_map[(yr, eid)],
            "full_file": full_map[(yr, eid)],
            "params_file": params_map[(yr, eid)],
        })

    return triplets


def compute_gap_mask(t, threshold=50):
    dt = np.diff(t)
    g_mask = dt <= threshold
    g_mask = np.concatenate([[True], g_mask])
    return g_mask

Using device: cpu


In [2]:
def robust_load_dat(file_path):
    """
    Robust loader for .dat files:
    - supports files with a header row like 'col1 col2 ...'
    - supports whitespace/comma separated text
    - keeps only numeric rows
    - returns at least the first two numeric columns
    """
    # First try: pandas automatic parsing
    try:
        df = pd.read_csv(
            file_path,
            sep=r"\s+|,",
            engine="python",
            comment="#",
            header=None
        )

        # convert everything to numeric; non-numeric becomes NaN
        df = df.apply(pd.to_numeric, errors="coerce")

        # drop rows that are completely non-numeric
        df = df.dropna(how="all")

        # keep rows where first two columns are numeric
        if df.shape[1] < 2:
            raise ValueError(f"{file_path} has fewer than 2 columns.")

        df = df.dropna(subset=[0, 1])

        data = df.iloc[:, :2].to_numpy(dtype=float)

        if len(data) == 0:
            raise ValueError(f"{file_path} has no usable numeric data rows.")

        return data

    except Exception as e1:
        # Fallback: try skipping first row manually
        try:
            data = np.loadtxt(file_path, skiprows=1)
            if data.ndim == 1:
                data = data.reshape(1, -1)
            if data.shape[1] < 2:
                raise ValueError(f"{file_path} has fewer than 2 numeric columns.")
            return data[:, :2].astype(float)

        except Exception as e2:
            raise ValueError(
                f"Could not read numeric data from {file_path}. "
                f"Pandas error: {e1}; loadtxt fallback error: {e2}"
            )

In [3]:
# =========================
# CELL 1 
# Physics functions + params reader
# =========================
def magnification_torch(u):
    eps = 1e-6
    u = torch.clamp(u, min=eps)
    return (u**2 + 2.0) / (u * torch.sqrt(u**2 + 4.0))


def microlensing_mag_torch(t, t0, tE, u0, I0, fbl):
    u = torch.sqrt(u0**2 + ((t - t0) / tE)**2)
    A = magnification_torch(u)
    flux_ratio = fbl * A + (1.0 - fbl)
    mag = I0 - 2.5 * torch.log10(flux_ratio)
    return mag


def read_params_file(params_file):
    """
    Read OGLE-style params file, e.g.

      Tmax   2458228.980     0.049
      tau         54.471     0.924
      umin         0.412     0.011
      fbl          0.754     0.027
      I0          17.245     0.039

    We only use the central value (second column).
    """
    wanted = {"Tmax", "tau", "umin", "I0", "fbl"}
    found = {}

    with open(params_file, "r") as f:
        lines = f.readlines()

    for line in lines:
        parts = line.strip().replace(",", " ").split()

        if len(parts) < 2:
            continue

        key = parts[0]

        if key in wanted:
            try:
                value = float(parts[1])
                found[key] = value
            except ValueError:
                continue

    missing = wanted - set(found.keys())
    if missing:
        raise ValueError(
            f"Missing parameters {sorted(missing)} in {params_file}. "
            f"Found only: {sorted(found.keys())}"
        )

    return {
        "Tmax": found["Tmax"],
        "tau": found["tau"],
        "umin": found["umin"],
        "I0": found["I0"],
        "fbl": found["fbl"],
    }

In [4]:
# =========================
# CELL 2
# Prepare one event for fitting
# =========================
def prepare_single_event(cutoff_file, full_file):
    cutoff_data = robust_load_dat(cutoff_file)
    full_data = robust_load_dat(full_file)
    if cutoff_data.ndim == 1:
        cutoff_data = cutoff_data.reshape(1, -1)
    if full_data.ndim == 1:
        full_data = full_data.reshape(1, -1)

    t_obs = cutoff_data[:, 0]
    m_obs = cutoff_data[:, 1]

    t_all = full_data[:, 0]
    m_all = full_data[:, 1]

    cutoff_idx = len(t_obs)

    if cutoff_idx >= len(t_all):
        raise ValueError(f"Cutoff file {cutoff_file} has no future points relative to {full_file}.")

    if not np.allclose(t_obs, t_all[:cutoff_idx], atol=1e-8):
        raise ValueError(
            f"Cutoff file {os.path.basename(cutoff_file)} is not the prefix of {os.path.basename(full_file)}."
        )

    t_future = t_all[cutoff_idx:]
    m_future = m_all[cutoff_idx:]

    gap_mask = compute_gap_mask(t_obs)
    t_fit = t_obs.copy()
    m_fit = m_obs.copy()

    return {
        "t_obs": t_obs,
        "m_obs": m_obs,
        "t_future": t_future,
        "m_future": m_future,
        "t_all": t_all,
        "m_all": m_all,
        "cutoff_idx": cutoff_idx,
        "gap_mask": gap_mask,
        "t_fit": t_fit,
        "m_fit": m_fit,
    }

In [5]:
# =========================
# CELL 3
# Simple parameter-fitting model
# =========================
class SimpleMicrolensingFitter(nn.Module):
    def __init__(self, t_init, m_init):
        super().__init__()

        # robust initialization from edge points
        n_front = max(5, len(m_init) // 5)
        n_back = max(5, len(m_init) // 5)
        edge_points = np.concatenate([m_init[:n_front], m_init[-n_back:]])
        I0_guess = float(np.median(edge_points))

        # peak time from brightest point (smallest magnitude)
        t0_guess = float(t_init[np.argmin(m_init)])

        # crude event-width estimate
        deviation = np.abs(m_init - I0_guess)
        thresh = max(0.03, 2.0 * np.std(edge_points))
        active_idx = np.where(deviation > thresh)[0]
        if len(active_idx) >= 2:
            event_width = float(t_init[active_idx[-1]] - t_init[active_idx[0]])
            tE_guess = max(event_width, 30.0)
        else:
            tE_guess = max((t_init[-1] - t_init[0]) / 4.0, 30.0)

        amp = max(I0_guess - float(np.min(m_init)), 0.02)
        u0_guess = 0.3 if amp < 0.3 else 0.1
        fbl_guess = 0.8

        self.t0 = nn.Parameter(torch.tensor(t0_guess, dtype=torch.float32))
        self.log_tE = nn.Parameter(torch.log(torch.tensor(tE_guess, dtype=torch.float32)))
        self.log_u0 = nn.Parameter(torch.log(torch.tensor(u0_guess, dtype=torch.float32)))
        self.I0 = nn.Parameter(torch.tensor(I0_guess, dtype=torch.float32))

        fbl_guess = np.clip(fbl_guess, 0.051, 0.989)
        fbl_raw_guess = np.log((fbl_guess - 0.05) / (0.99 - fbl_guess))
        self.fbl_raw = nn.Parameter(torch.tensor(fbl_raw_guess, dtype=torch.float32))

    def forward(self, t):
        tE = torch.exp(self.log_tE) + 1e-4
        u0 = torch.exp(self.log_u0) + 1e-4
        fbl = 0.05 + 0.94 * torch.sigmoid(self.fbl_raw)
        return microlensing_mag_torch(t, self.t0, tE, u0, self.I0, fbl)

    def get_physical_params(self):
        tE = (torch.exp(self.log_tE) + 1e-4).item()
        u0 = (torch.exp(self.log_u0) + 1e-4).item()
        fbl = (0.05 + 0.94 * torch.sigmoid(self.fbl_raw)).item()
        return {
            "Tmax": self.t0.item(),
            "tau": tE,
            "umin": u0,
            "I0": self.I0.item(),
            "fbl": fbl,
        }

In [6]:
# =========================
# CELL 4
# Fit one event
# =========================
def fit_single_event(event_dict, num_epochs=3000, lr=0.01, verbose=False):
    t_fit_np = event_dict["t_fit"]
    m_fit_np = event_dict["m_fit"]

    t_fit = torch.tensor(t_fit_np, dtype=torch.float32, device=device)
    m_fit = torch.tensor(m_fit_np, dtype=torch.float32, device=device)

    model = SimpleMicrolensingFitter(t_fit_np, m_fit_np).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    n = len(t_fit_np)
    weights = np.ones(n, dtype=np.float32)

    # emphasize front baseline, peak region, and end region near cutoff
    front_n = max(3, int(0.2 * n))
    weights[:front_n] *= 2.0

    baseline_est = float(np.median(np.concatenate([m_fit_np[:max(5, n // 5)], m_fit_np[-max(5, n // 5):]])))
    amp_est = np.maximum(baseline_est - m_fit_np, 0.0)
    amp_norm = amp_est / (np.max(amp_est) + 1e-6)
    weights *= (1.0 + 2.0 * amp_norm)

    end_n = max(3, int(0.15 * n))
    weights[-end_n:] *= 2.0

    weights = torch.tensor(weights, dtype=torch.float32, device=device)

    loss_history = []

    for epoch in range(num_epochs):
        optimizer.zero_grad()

        pred = model(t_fit)
        mse_loss = torch.mean(weights * (pred - m_fit) ** 2)

        params = model.get_physical_params()
        tE = torch.tensor(params["tau"], dtype=torch.float32, device=device)
        u0 = torch.tensor(params["umin"], dtype=torch.float32, device=device)

        reg_loss = 1e-6 * (tE ** 2) + 1e-4 * (u0 ** 2)
        loss = mse_loss + reg_loss

        loss.backward()
        optimizer.step()

        loss_history.append(loss.item())

        if verbose and (epoch + 1) % 500 == 0:
            print(f"Epoch {epoch+1}/{num_epochs}, loss = {loss.item():.6f}")

    fitted_params = model.get_physical_params()
    return model, fitted_params, loss_history

In [7]:
# =========================
# CELL 5
# Evaluate event WITHOUT plotting
# =========================
@torch.no_grad()
def evaluate_event(model, event_dict):
    """
    Evaluates the model's performance on both future points and the full light curve.
    Now includes R-squared (coefficient of determination) calculation.
    """
    t_future = event_dict["t_future"]
    m_future = event_dict["m_future"]
    t_all = event_dict["t_all"]
    m_all = event_dict["m_all"]

    # Convert to tensors for model prediction
    t_future_tensor = torch.tensor(t_future, dtype=torch.float32, device=device)
    t_all_tensor = torch.tensor(t_all, dtype=torch.float32, device=device)

    pred_future = model(t_future_tensor).cpu().numpy()
    pred_all = model(t_all_tensor).cpu().numpy()

    # Define R-squared helper
    def calc_r2(y_true, y_pred):
        ss_res = np.sum((y_true - y_pred) ** 2) # Sum of Squares of Residuals
        ss_tot = np.sum((y_true - np.mean(y_true)) ** 2) # Total Sum of Squares
        return 1 - (ss_res / (ss_tot + 1e-9)) # Avoid division by zero

    # Basic metrics
    future_mse = float(np.mean((pred_future - m_future) ** 2))
    future_mae = float(np.mean(np.abs(pred_future - m_future)))
    future_r2  = calc_r2(m_future, pred_future)

    full_curve_mse = float(np.mean((pred_all - m_all) ** 2))
    full_curve_mae = float(np.mean(np.abs(pred_all - m_all)))
    full_r2        = calc_r2(m_all, pred_all)

    return {
        "future_mse": future_mse,
        "future_mae": future_mae,
        "future_r2": future_r2,
        "full_curve_mse": full_curve_mse,
        "full_curve_mae": full_curve_mae,
        "full_r2": full_r2
    }

In [8]:
# =========================
# CELL 6
# Build comparison row + summary tables
# =========================
def build_comparison_row(year, event_id, cutoff_file, full_file, params_file,
                         fitted_params, true_params, metrics):
    """
    Constructs a dictionary containing all metrics and parameter comparisons 
    for a single microlensing event.
    """
    row = {
        "year": year,
        "event_id": event_id,
        "future_mse": metrics["future_mse"],
        "future_mae": metrics["future_mae"],
        "future_r2": metrics["future_r2"],
        "full_r2": metrics["full_r2"],
    }

    # Physical parameters to track
    mapping = [("Tmax", "Tmax"), ("tau", "tau"), ("umin", "umin"), ("I0", "I0"), ("fbl", "fbl")]

    for true_key, pred_key in mapping:
        true_val = true_params[true_key]
        pred_val = fitted_params[pred_key]
        row[f"{true_key}_true"] = true_val
        row[f"{true_key}_pred"] = pred_val
        # Resid (Residual): Predicted - True. Shows if we are over or underestimating.
        row[f"{true_key}_resid"] = pred_val - true_val
        # Abs_err: Magnitude of error.
        row[f"{true_key}_abs_err"] = abs(pred_val - true_val)

    return row

def build_summary_tables(results_df):
    """
    Aggregates results into descriptive statistics (mean, median, std, min, max).
    """
    if len(results_df) == 0:
        return pd.DataFrame(), pd.DataFrame()

    # Select columns that represent errors and goodness-of-fit
    resid_cols = [c for c in results_df.columns if c.endswith("_resid")]
    abs_err_cols = [c for c in results_df.columns if c.endswith("_abs_err")]
    metric_cols = ["future_mse", "future_mae", "future_r2", "full_r2"]
    
    summary_cols = resid_cols + abs_err_cols + metric_cols

    # Calculate overall stats
    overall_summary = results_df[summary_cols].agg(["mean", "median", "std", "min", "max"]).T
    overall_summary = overall_summary.reset_index().rename(columns={"index": "metric"})

    # Calculate mean metrics grouped by year
    by_year_summary = results_df.groupby("year")[summary_cols].mean().reset_index()

    return overall_summary, by_year_summary

In [9]:
# =========================
# CELL 7
# Batch main loop: run all matched events and output tables
# =========================
def run_batch_parameter_table():
    triplets = build_event_triplets(
        use_one_folder=USE_ONE_FOLDER,
        base_dir=BASE_DIR,
        cutoff_dir=CUTOFF_DIR,
        full_dir=FULL_DIR,
        params_dir=PARAMS_DIR,
        year=YEAR,
        limit=LIMIT,
    )

    print(f"Matched complete triplets: {len(triplets)}")

    if len(triplets) == 0:
        print("No complete matched event triplets found.")
        print("Check your file names and folder paths.")
        return None, None, None

    rows = []
    failed = []

    for i, triplet in enumerate(triplets, start=1):
        year = triplet["year"]
        event_id = triplet["event_id"]
        cutoff_file = triplet["cutoff_file"]
        full_file = triplet["full_file"]
        params_file = triplet["params_file"]

        try:
            event_dict = prepare_single_event(cutoff_file, full_file)

            model, fitted_params, loss_history = fit_single_event(
                event_dict,
                num_epochs=NUM_EPOCHS,
                lr=LR,
                verbose=False,
            )

            metrics = evaluate_event(model, event_dict)
            true_params = read_params_file(params_file)

            row = build_comparison_row(
                year=year,
                event_id=event_id,
                cutoff_file=cutoff_file,
                full_file=full_file,
                params_file=params_file,
                fitted_params=fitted_params,
                true_params=true_params,
                metrics=metrics,
            )
            rows.append(row)

            if i % VERBOSE_EVERY == 0 or i == 1 or i == len(triplets):
                print(f"[{i}/{len(triplets)}] done: year={year}, event={event_id}, future_MAE={metrics['future_mae']:.6f}")

        except Exception as e:
            failed.append({
                "year": year,
                "event_id": event_id,
                "error": str(e),
            })
            print(f"[FAILED] year={year}, event={event_id}: {e}")

    results_df = pd.DataFrame(rows)
    overall_summary, by_year_summary = build_summary_tables(results_df)

    if len(results_df) > 0:
        results_df = results_df.sort_values(["year", "event_id"]).reset_index(drop=True)
        results_df.to_csv(MAIN_TABLE_CSV, index=False)

    if len(overall_summary) > 0:
        overall_summary.to_csv(OVERALL_SUMMARY_CSV, index=False)

    if len(by_year_summary) > 0:
        by_year_summary.to_csv(YEAR_SUMMARY_CSV, index=False)

    print("\nSaved files:")
    print("Main table:        ", MAIN_TABLE_CSV)
    print("Overall summary:   ", OVERALL_SUMMARY_CSV)
    print("By-year summary:   ", YEAR_SUMMARY_CSV)

    if len(failed) > 0:
        failed_df = pd.DataFrame(failed)
        failed_csv = os.path.join(OUTPUT_DIR, "failed_events.csv")
        failed_df.to_csv(failed_csv, index=False)
        print("Failed events log: ", failed_csv)

    return results_df, overall_summary, by_year_summary


# Actually run
results_df, overall_summary, by_year_summary = run_batch_parameter_table()

print("\n" + "="*40)
print("     CORE FITTING RESULTS (TOP 10)     ")
print("="*40)

if results_df is not None and len(results_df) > 0:
    # Filter to the most important columns for a quick glance
    important_cols = ["year", "event_id", "future_r2", "full_r2", "tau_resid", "umin_resid", "I0_resid"]
    # round(4) keeps the table clean
    print(results_df[important_cols].head(10).round(4).to_string(index=False))
else:
    print("No results to display.")

print("\n" + "="*40)
print("     AGGREGATE STATISTICAL SUMMARY     ")
print("="*40)

if overall_summary is not None:
    # Print with higher precision (5) for summary stats
    print(overall_summary.round(5).to_string(index=False))

Matched complete triplets: 10
[1/10] done: year=2024, event=012, future_MAE=0.088134
[10/10] done: year=2024, event=184, future_MAE=0.035682

Saved files:
Main table:         /Users/skyxu/Downloads/parameter_comparison_table.csv
Overall summary:    /Users/skyxu/Downloads/parameter_summary_overall.csv
By-year summary:    /Users/skyxu/Downloads/parameter_summary_by_year.csv

     CORE FITTING RESULTS (TOP 10)     
 year event_id  future_r2  full_r2  tau_resid  umin_resid  I0_resid
 2024      012     0.0075   0.9457     8.3264     -0.0094    0.0106
 2024      031     0.8464   0.8478    -1.3443     -0.3374   -0.0013
 2024      053     0.7166   0.8966     8.2806     -0.2641    0.0020
 2024      056     0.9188   0.9235    -1.3620      0.0140   -0.6387
 2024      067     0.7708   0.9821     9.6256     -0.1192   -0.1319
 2024      088     0.2566   0.8208    -2.4881      0.0057   -0.4997
 2024      131     0.0076   0.8500   -29.2080      0.2338   -2.7761
 2024      156     0.9231   0.9790    19